# 使用mobilenetv2根据数据集[年龄、性别和种族（人脸数据）](https://aistudio.baidu.com/aistudio/datasetdetail/106996)进行性别分类

## 数据集介绍
[年龄、性别和种族（人脸数据）](https://aistudio.baidu.com/aistudio/datasetdetail/106996)包含了三个种族，两个性别，年龄从0到99分布的23705个人脸样本。每个人脸样本都通过一组数组进行存储，并且可以转化为48×48的灰度图像。

# 代码

## 数据预处理

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import random
# 生成图像和对应标签
df=pd.read_csv('data/data106996/age_gender.csv')

! rm -r -f img
! rm -r -f train
print("正在读取数据集")
def pixels2image(pixels): 
    return np.array(pixels.split()).astype(np.float).reshape(48, 48)

# 1-99岁，0-男，1-女 共198类
imagesdatalist = []
imageslabellist = []
valimagesdatalist = []
valimageslabellist = []
for i in range(len(df)):
    pixels=df['pixels'][i]
    img = pixels2image(pixels)
    gender = df['gender'][i]
    age = df['age'][i]
    if age<100:
        al =age//5
    # elif age<=90:
    #     al = 16
    # elif age <= 100:
    #     al = 17
    else:
        al = 20
    cla = (al)*2 + gender
    if random.randint(0, 10) == 1:
        valimagesdatalist.append(img)
        valimageslabellist.append(cla)
    else:
        imagesdatalist.append(img)
        imageslabellist.append(cla)
    # img=Image.fromarray(img)
    # img.convert('L').save("train/"+str(cla)+"s"+str(i)+'.jpg')


正在读取数据集


/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/ipykernel_launcher.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  if sys.path[0] == '':


In [ ]:
# 生成paddle数据集
%cd /home/aistudio

import os
import cv2
import numpy as np
from paddle.io import Dataset
# 生成label
from matplotlib import pyplot as plt

class MyDataset(Dataset):
    """
    步骤一：继承 paddle.io.Dataset 类
    """
    def __init__(self, datalist,labellist, transform=None):
        """
        步骤二：实现 __init__ 函数，初始化数据集，将样本和标签映射到列表中
        """
        super(MyDataset, self).__init__()
        self.data_list = []
        for i,image in enumerate(datalist):
            # print(image.shape)
            image = np.array([image]).astype('uint8')
            image = np.transpose(image, (1,2,0))
            # image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            # 调整到可以进入该模型输入的大小
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            label=labellist[i]
            # image = cv2.resize(image, (48, 48))
            self.data_list.append([image, label])
        print(len(self.data_list))
        # 传入定义好的数据处理方法，作为自定义数据集类的一个属性
        self.transform = transform

    def __getitem__(self, index):
        """
        步骤三：实现 __getitem__ 函数，定义指定 index 时如何获取数据，并返回单条数据（样本数据、对应的标签）
        """
        # 根据索引，从列表中取出一个图像
        image, label = self.data_list[index]        

        # 将图片尺寸缩放道 224x224
        
        # 读入的图像数据格式是[H, W, C]
        # 使用转置操作将其变成[C, H, W]
        

        # # image = cv2.imread(image_path)
        if self.transform is not None:
            image = self.transform(image)
        # # 飞桨训练时内部数据格式默认为float32，将图像数据格式转换为 float32
        # image = image.astype('float32')
        # image = image.transpose(2, 0, 1)
        # # 应用数据处理方法到图像上
        
        image = np.transpose(image, (2,0,1))
        image = image.astype('float32')
        # 将数据范围调整到[-1.0, 1.0]之间
        image = image / 255.
        image = image * 2.0 - 1.0

        # CrossEntropyLoss要求label格式为int，将Label格式转换为 int
        label = int(label)
        # 返回图像和对应标签
        return image, label
    def show_image(self,id):
        image = self.data_list[id][0]
        plt.imshow(image)
    def set_len(self,lenth):
        self.data_list = self.data_list[:lenth]
    def __len__(self):
        """
        步骤四：实现 __len__ 函数，返回数据集的样本总数
        """
        return len(self.data_list)
print("preparing datasets")
from paddle.vision.transforms import Compose, RandomRotation,Resize,Normalize,RandomHorizontalFlip,ColorJitter
transform = Compose([ColorJitter(0.6, 0.5, 0.5, 0.4),RandomRotation(15),RandomHorizontalFlip(0.5)])
# transform = Normalize(mean=[127.5], std=[127.5], data_format='CHW')
print(len(imageslabellist),len(valimageslabellist),len(set(imageslabellist)),len(set(valimageslabellist)))
num =len(imagesdatalist)
# 打印数据集样本数        
train_dataset = MyDataset(imagesdatalist,imageslabellist,transform)
val_dataset = MyDataset(valimagesdatalist,valimageslabellist,transform)
train_dataset.show_image(0)
# val_dataset.show_image(0)
print('train_custom_dataset images: ',len(train_dataset), 'test_custom_dataset images: ',len(val_dataset))

In [ ]:
import paddle
from paddle.vision.models import MobileNetV2,mobilenet_v2,resnet50,LeNet,mobilenet_v1
classnum =len(set(valimageslabellist))
print(set(valimageslabellist),classnum)
print('飞桨框架内置模型：', paddle.vision.models.__all__)
network =paddle.vision.models.mobilenet_v2(pretrained=True,num_classes=42)
# network = MobileNetV2(num_classes=7)
model = paddle.Model(network)
# model.summary((None, 3, 48, 48))
# model = MobileNetV2(pretrained=True,num_classes=7)

In [ ]:
#训练
# 指定在 CPU/GPU 上训练
!mkdir checkpoint
import os
if os.path.exists("checkpoint/mobilenet_v2.pdopt"):  
    print("reload model")
    model.load('checkpoint/mobilenet_v2')
# 指定在 GPU 第 0 号卡上训练
# paddle.device.set_device('cpu')
paddle.device.set_device('gpu:0')
scheduler = paddle.optimizer.lr.CosineAnnealingDecay(learning_rate=0.5, T_max=10, verbose=True)#自动学习率,暂时不用
opt = paddle.optimizer.Momentum(learning_rate=0.0001, momentum=0.9, 
        weight_decay=paddle.regularizer.L2Decay(coeff=.00002),
        parameters=model.parameters())
# opt = paddle.optimizer.Adam(learning_rate=0.0003,parameters=model.parameters())
model.prepare(opt,
              paddle.nn.CrossEntropyLoss(),
              paddle.metric.Accuracy())#设置模型训练方式

class SelfDefineCallback(paddle.callbacks.VisualDL):# 自定义会带哦
    def __init__(self):
        super().__init__(log_dir='./log_Res101_sszq')
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        model.save('checkpoint/mobilenet_v2')
        os.system("echo 'epoch: {}' >> log.txt".format(epoch))


    def on_epoch_begin(self, epoch, logs=None):
        super().on_epoch_begin(epoch, logs)
callback=SelfDefineCallback()
model.fit(train_dataset,# 训练数据集
          val_dataset,# 评估数据集
          epochs=2000,# 总的训练轮次
          batch_size=32,# 批次计算的样本量大小
          num_workers=4,
          verbose=1,# 日志展示格式
          shuffle=True,# 是否打乱样本集
          callbacks=callback)